### Setup

In [1]:
import pandas as pd
import numpy as np
import random

import torch
import torch.nn as nn
import torch.nn.functional as F

import os
import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from common.utils import DataPreprocessor, set_seed

# from dotenv import load_dotenv
# load_dotenv(dotenv_path="../.env")
# MLFLOW_SERVICE_URI = os.getenv("MLFLOW_SERVICE_URI", "")

MOVIELENS_DATA_DIR = "../datasets/movielens-2k/user_ratedmovies.dat"
RANDOM_SEED = 42
BATCH_SIZE = 1024
EMB_DIM = 64
LR = 1e-3
EPOCHS = 5

set_seed(RANDOM_SEED)


/media/emma/10TB/home/bilab_archive/Bai/DPRecSys/.venv/lib/python3.11/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/media/emma/10TB/home/bilab_archive/Bai/DPRecSys/.venv/lib/python3.11/site-packages/torch/__config__.py:10: UserWarning: CUDA initialization: Unexpected error from cudaGetDeviceCount(). Did you run some cuda functions before calling NumCudaDevices() that might have already set an error? Error 804: forward compatibility was attempted on non supported HW (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:109.)
  return torch._C._show_config()
Seed set to 42


random seed set to 42
numpy seed set to 42
torch seed set to 42
lightning seed set to 42


### Load and Process DataFrame

In [2]:
interaction_df = DataPreprocessor().load_and_process_df(
    file_dir=MOVIELENS_DATA_DIR,
    year_range=(2006, 2008),
)
NUM_USER = interaction_df["userID"].nunique()
NUM_ITEM = interaction_df["movieID"].nunique()
interaction_df.head()

Data count: 855598
Data count after filtering by year (2006, 2008): 480608
Num of distinct users: 2103
Num of distinct items: 9519
done!
------------------------------
Filtering by min user/item interactions (0/0):
Data count before: 480608
Data count after: 480608
done!
------------------------------
Re-index mapping dumped into ...
user: ../datasets/userid_mapping.csv
item: ../datasets/itemid_mapping.csv
done!
------------------------------
==== Final Data Info: ====
Data Year Range: (2006, 2008)
Rating Threshold: 4.0
Num of interactions: 480608
Num of distinct users: 2103
Num of distinct items: 9519


,userID,movieID,rating,date_day,date_month,date_year,date_hour,date_minute,date_second,timestamp,label
0,0,2,1.0,29,10,2006,23,17,16,2006-10-29 23:17:16,0
1,0,31,4.5,29,10,2006,23,23,44,2006-10-29 23:23:44,1
2,0,98,4.0,29,10,2006,23,30,8,2006-10-29 23:30:08,1
3,0,141,2.0,29,10,2006,23,16,52,2006-10-29 23:16:52,0
4,0,144,4.0,29,10,2006,23,29,30,2006-10-29 23:29:30,1


### Prepare Train/Valid/Test Set

In [3]:
# TODO: determine which method to use for splitting
# 1. Split by year
# 2. Stratified split by user, timestamp

train_df, valid_df, test_df = DataPreprocessor().stratified_time_split(
    interaction_df,
    time_col="timestamp",
    train_ratio=0.75,
    val_ratio=0.1,
    test_ratio=0.15,
)

Splitting data into train/valid/test by time period with ratio=(0.75 : 0.1 : 0.15):
train: 359669 (74.84%
valid: 47149 (9.81%)
test: 73790 (15.35%)
------------------------------ 

Check target label distribution after splitting (%):
train label
0    0.556042
1    0.443958
Name: proportion, dtype: float64
valid label
0    0.611042
1    0.388958
Name: proportion, dtype: float64
test label
0    0.585526
1    0.414474
Name: proportion, dtype: float64


### Prepare Additional Data for Train/Inference

#### Build bi-partite graph for training

In [4]:
# NOTE: At training, we use interaction graph from train_df for train and validation
train_graph = DataPreprocessor().create_interaction_graph(train_df)

# NOTE: At inference, we can use graph of (train_df + valid_df)
# train_valid_graph = utils.create_interaction_graph(pd.concat([train_df, valid_df], axis=0))

Creating interaction graph...
Drop negative samples
  Num of all interactions: 359669
  Num of positive interactions: 159678 

Building edges...
Building bi-directed edges (duplications)...
Building labels...
Building bi-directed labels (duplications)... 

Interaction Graph: Data(edge_index=[2, 319356], edge_label=[319356])
Edge Index: tensor([[   0,    0,    0,  ..., 2953,  776, 3094],
        [1147, 1234,  702,  ..., 2102, 2102, 2102]])


#### Prepare prediction pool for inference/testing

In [5]:
# NOTE: Prepare prediction pool to evaluate the model
prediction_pool_df = DataPreprocessor().prepare_prediction_df(test_df, K=500)
prediction_pool_df.tail()

Prediction DataFrame:
User Pool: 2103
Item Pool: 7626, negative sampled to 500 items for each user
Num of interactions: 2103(users) * 500(items) = 1051500


,userID,movieID,label
1051495,2102,774,0
1051496,2102,3906,0
1051497,2102,1021,0
1051498,2102,1326,0
1051499,2102,4167,0


### Prepare DataLoader

In [6]:
# TODO: determine which Dataset to use
from common.datasets import UserItemPairDataset
from torch.utils.data import DataLoader

train_dataset = UserItemPairDataset(train_df)
valid_dataset = UserItemPairDataset(valid_df)
test_dataset = UserItemPairDataset(prediction_pool_df)
print("train data count:", len(train_dataset))
print("valid data count:", len(valid_dataset))
print("test data count:", len(test_dataset))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)


train data count: 359669
valid data count: 47149
test data count: 1051500


### Configure Model (LightningModule)

In [7]:
from models.gcn_rec import GCNRecID

model = GCNRecID(
    edge_index=train_graph.edge_index,  # shape [2, num_edges]
    num_user=NUM_USER,
    num_item=NUM_ITEM,
    dim_id=EMB_DIM,
    lr=LR,
)

### Configure Trainer and Experiment

In [8]:
from common._mlflow import get_mlflow_logger, get_callbacks

EXPERIMENT_NAME = "gcn-bce-exp"
RUN_NAME = "gcn-baseline-test12"
mlflow_logger = get_mlflow_logger(experiment_name=EXPERIMENT_NAME, run_name=RUN_NAME)
trainer_callbacks = get_callbacks(RUN_NAME, patience=3)  

In [9]:
from pytorch_lightning import Trainer

trainer = Trainer(
    max_epochs=EPOCHS,
    logger=mlflow_logger,
    log_every_n_steps=1,
    callbacks=trainer_callbacks,
    accelerator='auto',  # or 'auto', 'gpu'
    # devices=[0], # if gpu is available
)


/media/emma/10TB/home/bilab_archive/Bai/DPRecSys/.venv/lib/python3.11/site-packages/torch/cuda/__init__.py:734: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


### Train Model

In [10]:
# Start training
trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=valid_loader)


/media/emma/10TB/home/bilab_archive/Bai/DPRecSys/.venv/lib/python3.11/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:654: Checkpoint directory /media/emma/10TB/home/bilab_archive/Bai/DPRecSys/basic/test_checkpoints exists and is not empty.

  | Name  | Type   | Params | Mode 
-----------------------------------------
0 | model | GCN_ID | 793 K  | train
-----------------------------------------
793 K     Trainable params
0         Non-trainable params
793 K     Total params
3.173     Total estimated model params size (MB)
13        Modules in train mode
0         Modules in eval mode
/media/emma/10TB/home/bilab_archive/Bai/DPRecSys/.venv/lib/python3.11/site-packages/torch/cuda/__init__.py:734: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/media/emma/10TB/home/bilab_archive/Bai/DPRecSys/.venv/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.
/media/emma/10TB/home/bilab_archive/Bai/DPRecSys/.venv/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

/media/emma/10TB/home/bilab_archive/Bai/DPRecSys/.venv/lib/python3.11/site-packages/torch/cuda/__init__.py:734: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
/media/emma/10TB/home/bilab_archive/Bai/DPRecSys/.venv/lib/python3.11/site-packages/torch/cuda/__init__.py:734: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
/media/emma/10TB/home/bilab_archive/Bai/DPRecSys/.venv/lib/python3.11/site-packages/torch/cuda/__init__.py:734: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
/media/emma/10TB/home/bilab_archive/Bai/DPRecSys/.venv/lib/python3.11/site-packages/torch/cuda/__init__.py:734: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
/media/emma/10TB/home/bilab_archive/Bai/DPRecSys/.venv/lib/python3.11/site-packages/torch/cuda/__init__.py:734: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
/media/emma/10TB/home/bilab_archive/Bai/DPRecSys/.venv/lib/p

...

Validation: |          | 0/? [00:00<?, ?it/s]

Metric val_f1 improved by 0.010 >= min_delta = 0.0. New best score: 0.600
Epoch 3, global step 1408: 'val_f1' reached 0.59957 (best 0.59957), saving model to '/media/emma/10TB/home/bilab_archive/Bai/DPRecSys/basic/test_checkpoints/gcn-baseline-test12-best-checkpoint-epoch=03-val_f1=0.60.ckpt' as top 1
/media/emma/10TB/home/bilab_archive/Bai/DPRecSys/.venv/lib/python3.11/site-packages/torch/cuda/__init__.py:734: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
/media/emma/10TB/home/bilab_archive/Bai/DPRecSys/.venv/lib/python3.11/site-packages/torch/cuda/__init__.py:734: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
/media/emma/10TB/home/bilab_archive/Bai/DPRecSys/.venv/lib/python3.11/site-packages/torch/cuda/__init__.py:734: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
/media/emma/10TB/home/bilab_archive/Bai/DPRecSys/.venv/lib/python3.11/site-packages/torch/cuda/__init__.py:734: UserWarning: Can't in

Validation: |          | 0/? [00:00<?, ?it/s]

Metric val_f1 improved by 0.009 >= min_delta = 0.0. New best score: 0.609
Epoch 4, global step 1760: 'val_f1' reached 0.60857 (best 0.60857), saving model to '/media/emma/10TB/home/bilab_archive/Bai/DPRecSys/basic/test_checkpoints/gcn-baseline-test12-best-checkpoint-epoch=04-val_f1=0.61.ckpt' as top 1
`Trainer.fit` stopped: `max_epochs=5` reached.


🏃 View run gcn-baseline-test12 at: http://140.112.106.216:3683/#/experiments/2/runs/f84bb291aa23446bbc154796205a30cf
🧪 View experiment at: http://140.112.106.216:3683/#/experiments/2


### Inference

In [11]:
# NOTE: the inference model MUST be the same as the training model
best_model_path = "test_checkpoints/gcn-baseline-test11-best-checkpoint-epoch=23-val_f1=0.65.ckpt"
model = GCNRecID.load_from_checkpoint(
    checkpoint_path=best_model_path, 
    edge_index=train_graph.edge_index,
    num_user=NUM_USER,
    num_item=NUM_ITEM,
    dim_id=EMB_DIM,
    lr=LR,
)


/media/emma/10TB/home/bilab_archive/Bai/DPRecSys/.venv/lib/python3.11/site-packages/torch/cuda/__init__.py:734: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


In [12]:
# start inference
trainer.test(model=model, dataloaders=test_loader)

/media/emma/10TB/home/bilab_archive/Bai/DPRecSys/.venv/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


Testing: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │    0.5716319680213928     │
│          test_f1          │    0.08631569892168045    │
│         test_loss         │    2.4332122802734375     │
│         test_prec         │   0.046012409031391144    │
│         test_rec          │    0.6956578493118286     │
└───────────────────────────┴───────────────────────────┘

🏃 View run gcn-baseline-test12 at: http://140.112.106.216:3683/#/experiments/2/runs/f84bb291aa23446bbc154796205a30cf
🧪 View experiment at: http://140.112.106.216:3683/#/experiments/2


[{'test_loss': 2.4332122802734375,
  'test_acc': 0.5716319680213928,
  'test_prec': 0.046012409031391144,
  'test_rec': 0.6956578493118286,
  'test_f1': 0.08631569892168045}]